# Semenov et al. (2025), Figure 3C, from dandiset 001030 (data elsewhere)

Paper. Semenov I, Kim V, Silkuniene G, Pakhomov AG (2025), *Cell Reports Physical Science* 6,
102660, [doi:10.1016/j.xcrp.2025.102660](https://doi.org/10.1016/j.xcrp.2025.102660); the claims
file was written from the NIH author manuscript (PMC12320893) with its supplement.

Dataset. The claim fig3c-sustained-firing-2s has `data: elsewhere`, in
[DANDI:001030](https://dandiarchive.org/dandiset/001030/0.240520.1924), version 0.240520.1924, the
only published version, which the paper lists without a version ("High-frequency (2 kHz) sine wave
stimulation of individual hippocampal neurons: the effect of electric field"). This notebook tests
that one claim; the attempt's other claims are in `reproduce_semenov_2025.ipynb` or blocked (see
the attempt record).

| Claim | Tested here |
|---|---|
| fig3c-sustained-firing-2s | yes |

No random numbers are used.

In [ ]:
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import fastimg

pd.set_option("display.width", 200)
started = time.time()
sessions = fastimg.read_all(fastimg.list_assets())
print(f"{len(sessions)} sessions read from dandiset {fastimg.DANDISET_ID} version {fastimg.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

In [ ]:
info = pd.DataFrame([{k: s[k] for k in ("cell", "subject_id", "amplitude_v", "species", "rate_hz", "shape", "session_start")}
                     for s in sessions]).sort_values(["cell", "amplitude_v"]).reset_index(drop=True)
print(info.to_string(index=False))
print("\nSessions per cell:", dict(Counter(info.cell)))
print("Species:", dict(Counter(info.species)), "; rates:", dict(Counter(info.rate_hz)))
print("Frames per session:", dict(Counter(s["shape"][0] for s in sessions)),
      "; durations (s):", sorted({round(s["shape"][0] / fastimg.FS, 2) for s in sessions}))
first = np.array([s["frame_means"][0] / np.median(s["frame_means"][1:200]) for s in sessions])
print("First frame over the median of frames 1-199, range:", first.min().round(2), "to", first.max().round(2))
print("\nDescription of one session:\n", sessions[0]["description"])

The 17 sessions come from four cells (c5 to c8), each at several amplitudes; two sessions of cell
c5 stop at 1.91 s, before the 2.025-s end of the stimulus. The files give the species as *Mus
musculus*, where the paper used rat neurons, and the images are 16 x 16 pixels. The first frame is
several times brighter than the others in every file, and the next frames recover from it over
a few ms; the first 20 frames (6.4 ms) are dropped. Amplitudes are in volts (one
polarity), 36 V/cm per volt by the description.

Clock. Each file is one image series with no other stream; the stimulus timing (25 ms to
2.025 s) comes from the description. The check below shows the response starts with the
stimulus: the AP counts (reading b) in the 25 ms before and the 25 ms after the stated onset.

In [ ]:
for s in sessions:
    s["dF"] = fastimg.delta_f(s["frame_means"])
    s["peaks_raw"] = {r: fastimg.detect(s["dF"], r) for r in "ab"}
start = int(fastimg.STIM_START_S * fastimg.FS)
pre = sum(int(((p >= fastimg.DROP_FRAMES) & (p < start)).sum()) for s in sessions for p in [s["peaks_raw"]["b"]])
post = sum(int(((p >= start) & (p < 2 * start)).sum()) for s in sessions for p in [s["peaks_raw"]["b"]])
first_ap = [s["peaks_raw"]["b"][s["peaks_raw"]["b"] >= start][:1] for s in sessions]
print("APs (reading b) between the dropped frames and the onset (18.6 ms):", pre, "; in the 25 ms after:", post)
print("Latency of the first AP after the stated onset (ms), per session with APs:",
      sorted(round(1000 * (f[0] - start) / fastimg.FS) for f in first_ap if len(f)))

No AP precedes the stated onset, so the stimulus timing is consistent with the data (the first
APs come during the 250-ms ramp, as the claim describes).

## fig3c-sustained-firing-2s

> In the low-conductivity solution, a 2-s unmodulated 2-kHz sine wave ramped over 250 ms evoked
> firing for its whole duration, rising with field until replaced by a firing block with sustained
> depolarization.

Criterion: traces corrected for bleaching by subtracting a trace without APs (Methods); AP
detection readings (a) and (b). Reproduced when an intermediate field has APs in the last 250 ms of
the stimulus, the AP count rises with field up to some field, and the highest field has no APs
after 250 ms with mean dF above F0 by more than 5% over the rest of the stimulus. The criterion
does not say whether it applies per cell; the field series of each cell is judged on its own, as
Figure 3C shows one cell, and every cell is reported.

In [ ]:
lo, hi = fastimg.windows()["stimulus"]
counts = pd.DataFrame([{"subject_id": s["subject_id"],
                        **{r: int(((s["peaks_raw"][r] >= lo) & (s["peaks_raw"][r] < hi)).sum()) for r in "ab"}}
                       for s in sessions])
free = counts[(counts.a == 0) & (counts.b == 0)]
print("Sessions without APs during the stimulus under both readings (uncorrected):", free.subject_id.tolist())
reference = next(s for s in sessions if s["subject_id"] == free.subject_id.iloc[0])
w = fastimg.windows()
rows = []
for s in sessions:
    n = min(len(s["dF"]), len(reference["dF"]))
    for corrected in (True, False):
        x = s["dF"][:n] - (reference["dF"][:n] if corrected else 0)
        row = {"cell": s["cell"], "amplitude_v": s["amplitude_v"], "corrected": corrected}
        for r in "ab":
            p = fastimg.detect(x, r)
            for name, (lo, hi) in w.items():
                row[f"{r}_{name}"] = int(((p >= lo) & (p < min(hi, n))).sum())
        lo, hi = w["after_ramp"]
        row["mean_dF_after_ramp"] = np.nanmean(x[lo:min(hi, n)])
        row["frames"] = n
        rows.append(row)
table = pd.DataFrame(rows).sort_values(["corrected", "cell", "amplitude_v"], ascending=[False, True, True])
print(table.round(2).to_string(index=False))

In [ ]:
def judge(sub, r):
    sub = sub.sort_values("amplitude_v")
    counts = sub[f"{r}_after_ramp"].to_numpy()
    top = sub.iloc[-1]
    intermediate = sub.iloc[1:-1]
    late = bool((intermediate[f"{r}_last_250ms"] > 0).any())
    k = int(np.argmax(counts))
    rises = k > 0 and bool(np.all(np.diff(counts[:k + 1]) >= 0))
    block = top[f"{r}_after_ramp"] == 0 and top.mean_dF_after_ramp > 5
    return {"late_firing_at_intermediate_field": late, "count_rises_to_a_field": rises,
            "block_at_highest_field": bool(block), "holds": late and rises and bool(block),
            "highest_field_aps_after_ramp": int(top[f"{r}_after_ramp"]),
            "highest_field_mean_dF": round(float(top.mean_dF_after_ramp), 2)}
rows = []
for corrected in (True, False):
    for cell, sub in table[table.corrected == corrected].groupby("cell"):
        for r in "ab":
            rows.append({"corrected": corrected, "cell": cell, "reading": r, "fields": len(sub), **judge(sub, r)})
verdict = pd.DataFrame(rows)
print(verdict.to_string(index=False))
print("\nAP times (ms) of reading (b) after the ramp at each cell's highest field, bleach-corrected:")
lo, hi = fastimg.windows()["after_ramp"]
for cell in sorted(info.cell.unique()):
    s = max((s for s in sessions if s["cell"] == cell), key=lambda s: s["amplitude_v"])
    n = min(len(s["dF"]), len(reference["dF"]))
    p = fastimg.detect(s["dF"][:n] - reference["dF"][:n], "b")
    print(f"  {cell} at {s['amplitude_v']} V:", np.round(1000 * p[(p >= lo) & (p < hi)] / fastimg.FS).astype(int).tolist())

In [ ]:
cell = "c7"
subset = sorted([s for s in sessions if s["cell"] == cell], key=lambda s: s["amplitude_v"])
fig, axes = plt.subplots(len(subset), 1, figsize=(12, 1.6 * len(subset)), sharex=True)
lo_all, hi_all = np.inf, -np.inf
for ax, s in zip(axes, subset):
    n = min(len(s["dF"]), len(reference["dF"]))
    x = s["dF"][:n] - reference["dF"][:n]
    t = np.arange(n) / fastimg.FS
    ax.plot(t, x, lw=0.4)
    p = fastimg.detect(x, "b")
    ax.plot(t[p], x[p], "r.", ms=2)
    ax.set_ylim(-10, 25)
    ax.set_ylabel(f"{s['amplitude_v']} V", fontsize=8)
    lo_all, hi_all = min(lo_all, np.nanmin(x)), max(hi_all, np.nanmax(x))
axes[-1].set_xlabel("time (s)")
plt.show()
print(f"Cell {cell}, bleach-corrected dF range {lo_all:.1f} to {hi_all:.1f}% (axes -10 to 25%); red: APs of reading (b)")

**Result: partially reproduced.** In all four cells, under both readings with the bleaching
correction, an intermediate field fires in the last 250 ms of the stimulus and the AP count after
the ramp rises with field (without the correction, c5 under reading (a) has no late firing). In no cell does the highest field meet the block test as
written (no APs after the ramp and mean dF above 5%). Cell c7 comes closest: at 2.3 V, reading (b)
finds one AP after the ramp, at 1,707 ms, with a corrected mean dF of 6.66%; the other cells keep
firing at their highest field, with mean dF from 2.76 to 3.63%. Reading (a) counts the noise on a
depolarized plateau as APs (197 after the ramp in c7 at 2.3 V).

## Summary

| Claim | Outcome |
|---|---|
| fig3c-sustained-firing-2s | partially reproduced: sustained firing rising with field in all four cells, no block meeting the test in any |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: F is the whole 16 x 16 frame, as the soma cannot be
outlined at this resolution; the bleaching trace is the dataset's only session without APs (cell
c7 at 1.0 V), used for every cell; the judgement is per cell.

What was found about the dataset: the species is given as mouse for neurons the paper describes as
rat; the images are 16 x 16 pixels; two sessions end before the stimulus does; the first frame of
every file is an artefact.